# Streaming — 02: Streaming Writes to Iceberg (Exactly-Once)

## What you will learn
Combining Structured Streaming with Apache Iceberg gives you:
- **Exactly-once delivery** — no duplicates, no missing records
- **ACID guarantees** on every micro-batch commit
- **Time travel** on streaming data — query historical state of live tables
- **Schema evolution** while the stream is running
- **Compaction** of small streaming files without downtime

## Why Iceberg for streaming?

Traditional streaming sinks (Parquet files, Hive tables) have problems:
- No atomicity — partial writes are visible to readers
- Small files accumulate rapidly (one file per micro-batch per partition)
- No time travel — you cannot query "what did the stream look like at 14:00?"

Iceberg solves all of these with its snapshot model:
```
Micro-batch 1: events 1-100   → Snapshot 1 (atomic commit)
Micro-batch 2: events 101-200 → Snapshot 2 (atomic commit)
Micro-batch 3: events 201-300 → Snapshot 3 (atomic commit)

Reader always sees a consistent snapshot.
No partial batch is ever visible.
```

## Exactly-once guarantee
Spark Structured Streaming uses **checkpointing** to track which batches have
been written. Iceberg uses **two-phase commit**:
1. Write data files to storage
2. Commit snapshot atomically to metadata

If the job fails after step 1 but before step 2, the data files are orphaned
(not referenced by any snapshot) — they will be cleaned up by VACUUM.
The batch will be replayed on restart → exactly-once.


In [ ]:
import os, time, datetime, pathlib, shutil, random, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('streaming-iceberg')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

## Step 1 — Setup: Iceberg Table as Streaming Sink

We create an Iceberg table that will receive streaming data.
The table is partitioned by hour — a common choice for event streams
since it keeps each partition to a manageable size.


In [ ]:
# Setup directories
stream_input = f"{DATA_DIR}/stream_ice_input"
stream_ckpt  = f"{DATA_DIR}/stream_ice_checkpoint"

for d in [stream_input, stream_ckpt]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Create Iceberg sink table
spark.sql("CREATE DATABASE IF NOT EXISTS local.events_db")
spark.sql("DROP TABLE IF EXISTS local.events_db.clickstream")

spark.sql("""
    CREATE TABLE local.events_db.clickstream (
        event_id    BIGINT,
        user_id     BIGINT,
        event_type  STRING,
        page        STRING,
        device      STRING,
        country     STRING,
        revenue     DOUBLE,
        event_ts    TIMESTAMP,
        event_hour  INT
    )
    USING iceberg
    PARTITIONED BY (hours(event_ts))
    TBLPROPERTIES (
        'write.format.default'           = 'parquet',
        'write.parquet.compression-codec'= 'zstd',
        'write.metadata.compression-codec' = 'gzip'
    )
""")

print("Iceberg sink table created: local.events_db.clickstream")
spark.sql("DESCRIBE EXTENDED local.events_db.clickstream").show(30, truncate=False)

## Step 2 — Define the Streaming Reader

We read from a JSON file directory (same pattern as notebook 01).
In production this would be a Kafka topic.

Key difference from batch: `spark.readStream` instead of `spark.read`.


In [ ]:
event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("page",       StringType()),
    StructField("device",     StringType()),
    StructField("country",    StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_ts",   TimestampType()),
])

raw_stream = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input)
)

# Add derived column before writing
enriched_stream = raw_stream.withColumn(
    "event_hour", F.hour("event_ts")
)

print("Streaming DataFrame schema:")
enriched_stream.printSchema()
print(f"Is streaming: {enriched_stream.isStreaming}")

## Step 3 — Write Stream to Iceberg

`writeStream.format("iceberg")` writes each micro-batch as an atomic
Iceberg snapshot. The checkpoint tracks which batches are committed.

**Output mode:** `append` — each micro-batch appends new rows.
Iceberg also supports `complete` for aggregations (replaces table each batch).


In [ ]:
# Start the streaming write to Iceberg
query = (
    enriched_stream
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", stream_ckpt)
    .option("fanout-enabled", "true")    # allow out-of-order partition writes
    .toTable("local.events_db.clickstream")
)

print(f"Streaming query started: {query.name or 'unnamed'}")
print(f"Status: {query.status['message']}")

## Step 4 — Generate Events and Observe Iceberg Snapshots

Each batch of events creates a new Iceberg snapshot.
We can observe snapshots being created in real time.


In [ ]:
EVENTS    = ["page_view","click","purchase","search","add_to_cart"]
PAGES     = ["/home","/products","/cart","/checkout","/search"]
DEVICES   = ["mobile","desktop","tablet"]
COUNTRIES = ["US","UK","DE","FR","JP","CA"]
random.seed(42)

def generate_events(batch_id, n=30):
    now = datetime.datetime.now()
    events = []
    for i in range(n):
        ts = now - datetime.timedelta(seconds=random.randint(0, 300))
        events.append({
            "event_id":   batch_id * 1000 + i,
            "user_id":    random.randint(1, 500),
            "event_type": random.choice(EVENTS),
            "page":       random.choice(PAGES),
            "device":     random.choice(DEVICES),
            "country":    random.choice(COUNTRIES),
            "revenue":    round(random.uniform(10, 500), 2) if random.random() > 0.7 else 0.0,
            "event_ts":   ts.strftime("%Y-%m-%dT%H:%M:%S.%f")
        })
    path = f"{stream_input}/batch_{batch_id:04d}.json"
    with open(path, "w") as f:
        for e in events:
            f.write(pyjson.dumps(e) + "\n")
    return path

print("Generating event batches and watching Iceberg snapshots...")
for i in range(1, 5):
    path = generate_events(i)
    time.sleep(3)

    # Check Iceberg snapshots after each batch
    snaps = spark.sql("""
        SELECT snapshot_id, committed_at, operation,
               summary['added-records'] AS added_rows,
               summary['total-records'] AS total_rows
        FROM local.events_db.clickstream.snapshots
        ORDER BY committed_at
    """)
    snap_count = snaps.count()
    if snap_count > 0:
        print(f"  Batch {i}: {snap_count} Iceberg snapshot(s) committed")
        snaps.show(truncate=False)
    else:
        print(f"  Batch {i}: waiting for stream to process...")

time.sleep(5)
query.stop()
print("\nStream stopped.")

## Step 5 — Query the Iceberg Table (Batch + Time Travel)

Once data is in Iceberg, you can query it from any Spark session —
batch or streaming. Time travel works on streaming data too.


In [ ]:
# Batch query on the streamed data
total = spark.table("local.events_db.clickstream").count()
print(f"Total events in Iceberg: {total:,}")

print("\nEvents by type:")
spark.table("local.events_db.clickstream") \
     .groupBy("event_type") \
     .agg(F.count("*").alias("count"),
          F.sum("revenue").alias("total_revenue")) \
     .orderBy(F.desc("count")) \
     .show()

print("\nEvents by hour (partition distribution):")
spark.table("local.events_db.clickstream") \
     .groupBy("event_hour") \
     .count() \
     .orderBy("event_hour") \
     .show()

In [ ]:
# Time travel: query the table after the FIRST snapshot
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, summary['total-records'] AS total
    FROM local.events_db.clickstream.snapshots
    ORDER BY committed_at
""").collect()

if len(snapshots) >= 2:
    first_snap = snapshots[0]["snapshot_id"]
    last_snap  = snapshots[-1]["snapshot_id"]

    count_first = spark.read.option("snapshot-id", first_snap) \
                       .table("local.events_db.clickstream").count()
    count_last  = spark.table("local.events_db.clickstream").count()

    print(f"Snapshot {first_snap} (first batch) : {count_first:,} rows")
    print(f"Snapshot {last_snap}  (latest)       : {count_last:,} rows")
    print()
    print("Time travel on streaming data works exactly like batch Iceberg tables.")
    print("Every micro-batch commit = a queryable snapshot.")
else:
    print("Not enough snapshots yet — try running more batches.")

## Step 6 — Streaming Aggregation into Iceberg

Beyond simple append, you can write **streaming aggregations** to Iceberg.
This pattern is useful for building real-time dashboards backed by Iceberg.

We use `complete` output mode which rewrites the entire result table
on each micro-batch — Iceberg makes this atomic.


In [ ]:
# New stream for aggregated sink
stream_input2 = f"{DATA_DIR}/stream_ice_input2"
stream_ckpt2  = f"{DATA_DIR}/stream_ice_checkpoint2"
shutil.rmtree(stream_input2, ignore_errors=True)
shutil.rmtree(stream_ckpt2, ignore_errors=True)
pathlib.Path(stream_input2).mkdir(parents=True, exist_ok=True)
pathlib.Path(stream_ckpt2).mkdir(parents=True, exist_ok=True)

# Create aggregated sink table
spark.sql("DROP TABLE IF EXISTS local.events_db.event_summary")
spark.sql("""
    CREATE TABLE local.events_db.event_summary (
        event_type    STRING,
        country       STRING,
        event_count   BIGINT,
        total_revenue DOUBLE,
        avg_revenue   DOUBLE
    )
    USING iceberg
""")

raw_stream2 = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input2)
)

agg_stream = (
    raw_stream2
    .groupBy("event_type", "country")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("revenue").alias("avg_revenue")
    )
)

query2 = (
    agg_stream
    .writeStream
    .format("iceberg")
    .outputMode("complete")
    .option("checkpointLocation", stream_ckpt2)
    .toTable("local.events_db.event_summary")
)

print("Aggregated streaming query started.")

# Generate events and watch aggregation update
for i in range(1, 4):
    path = f"{stream_input2}/batch_{i:04d}.json"
    events = []
    for j in range(50):
        ts = datetime.datetime.now() - datetime.timedelta(seconds=random.randint(0, 60))
        events.append({
            "event_id": i*1000+j, "user_id": random.randint(1,100),
            "event_type": random.choice(EVENTS), "page": random.choice(PAGES),
            "device": random.choice(DEVICES), "country": random.choice(COUNTRIES),
            "revenue": round(random.uniform(10,500),2) if random.random() > 0.7 else 0.0,
            "event_ts": ts.strftime("%Y-%m-%dT%H:%M:%S.%f")
        })
    with open(path, "w") as f:
        for e in events: f.write(pyjson.dumps(e) + "\n")
    time.sleep(3)
    print(f"  Batch {i} processed")

time.sleep(4)
query2.stop()

print("\nCurrent aggregated state in Iceberg:")
spark.table("local.events_db.event_summary") \
     .orderBy(F.desc("total_revenue")) \
     .show(10)

print("\nNumber of snapshots (one per complete-mode micro-batch):")
spark.sql("SELECT COUNT(*) FROM local.events_db.event_summary.snapshots").show()

## Step 7 — Compaction: Solving the Small File Problem

Each streaming micro-batch creates small Parquet files.
After hours of streaming, you may have thousands of tiny files.

Iceberg's `rewrite_data_files` compacts them without stopping the stream.
A reader sees a consistent view throughout — the stream can keep writing
while compaction runs.


In [ ]:
# Show file count before compaction
print("Before compaction:")
spark.sql("""
    SELECT COUNT(*) AS file_count,
           SUM(file_size_in_bytes)/1024 AS total_kb,
           AVG(file_size_in_bytes)/1024 AS avg_kb_per_file
    FROM local.events_db.clickstream.files
""").show()

# Run compaction (can run concurrently with streaming writes)
print("Running compaction (safe to run while stream writes)...")
spark.sql("""
    CALL local.system.rewrite_data_files(
        table    => 'local.events_db.clickstream',
        strategy => 'binpack',
        options  => map('target-file-size-bytes', '67108864')
    )
""").show()

print("After compaction:")
spark.sql("""
    SELECT COUNT(*) AS file_count,
           SUM(file_size_in_bytes)/1024 AS total_kb,
           AVG(file_size_in_bytes)/1024 AS avg_kb_per_file
    FROM local.events_db.clickstream.files
""").show()

print("Compaction merged small streaming files into larger ones.")
print("Query performance on historical data improves significantly.")

## Summary: Streaming to Iceberg Best Practices

### Architecture pattern
```
Kafka / File Source
      │
      ▼ spark.readStream
Structured Streaming
      │ .writeStream.format("iceberg")
      ▼
Iceberg Table (append mode)
      │
      ├── Batch queries (analytics, BI)
      ├── Time travel (debugging, auditing)
      └── Periodic compaction (maintenance)
```

### Key configuration
```python
.writeStream
.format("iceberg")
.outputMode("append")          # or "complete" for aggregations
.option("checkpointLocation", "/path/checkpoint")
.option("fanout-enabled", "true")   # needed for unordered partition writes
.toTable("catalog.db.table")
```

### Compaction schedule
Run `rewrite_data_files` on a schedule (e.g., hourly):
```python
spark.sql("CALL catalog.system.rewrite_data_files(table => 'db.table')")
```
Safe to run concurrently with streaming writes — Iceberg's MVCC
ensures readers and writers never block each other.

### Exactly-once guarantee
- Spark checkpoint tracks committed batch IDs
- Iceberg two-phase commit ensures atomic snapshot creation
- On failure: incomplete files are orphaned, batch replays from checkpoint
- Result: no duplicates, no data loss

**Next:** `03_stateful_operations.ipynb` — flatMapGroupsWithState, session windows.
